# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Three archetypes, each with a trigger, a reason code, and an action label. The first two are
the transparent rules already validated in Weeks 4 and 6 (staleness, CTR-vs-position). The
third catches pages the honest model flags as higher decline risk that neither rule alone
catches — this is where the model earns its place over the rule baseline, and it's reported as
such, not hidden inside a single score.

| archetype | trigger | reason code | action |
|---|---|---|---|
| stale but visible | `days_since_last_update ≥ 180` and `impressions_90d ≥ 500` | `stale_but_visible` | `review_for_refresh` |
| good position, weak CTR | `position_tier` in {top_3, page_1} and `ctr` below that tier's median | `weak_ctr_good_position` | `review_for_ctr_fix` |
| model-flagged, rules silent | RF predicted decline probability in the top 20% and neither rule above fired | `model_flagged_decline_risk` | `review_general` |
| none of the above | — | `no_flag` | `no_action` |

Ranking within the queue is by the model's predicted decline probability — the same Random
Forest audited in Week 6, refit here on the **full** dataset (not the train/test split) because
this is the artifact meant for actual ranking use, not held-out evaluation. The Week-6 grouped-
split numbers (precision@K, ROC-AUC vs. the Week-4 baseline) remain the honest evidence for how
much to trust this ranking — restated in Section 2 — rather than re-measured on in-sample scores
here, which would overstate confidence.


In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
pd.set_option("display.width", 120)

if not os.path.exists("flyrank-ml-internship-starter"):
    get_ipython().system('git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git')
if os.path.basename(os.getcwd()) != "flyrank-ml-internship-starter":
    os.chdir("flyrank-ml-internship-starter")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- same leakage-free feature set as Weeks 5-6 ---
numeric_features = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
categorical_features = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

X_numeric = df[numeric_features].fillna(0)
X_categorical = pd.get_dummies(df[categorical_features].fillna("unknown"), prefix=categorical_features)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

# --- final model, fit on ALL rows: this is the deployed-ranking artifact, not the eval model ---
final_rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
final_rf.fit(X, y)
df["model_score"] = final_rf.predict_proba(X)[:, 1]

# ------------------------------------------------------------------
# Archetype rules (transparent, same thresholds as Weeks 4 and 6)
# ------------------------------------------------------------------
STALE_DAYS_THRESHOLD = 180
VISIBLE_IMPRESSIONS_THRESHOLD = 500
MODEL_TOP_PCT = 0.20

stale_visible = (
    (df["days_since_last_update"] >= STALE_DAYS_THRESHOLD)
    & (df["impressions_90d"] >= VISIBLE_IMPRESSIONS_THRESHOLD)
)

good_position = df["position_tier"].isin(["top_3", "page_1"])
median_ctr_by_tier = df.groupby("position_tier")["ctr"].transform("median")
weak_ctr = good_position & (df["ctr"] < median_ctr_by_tier)

model_threshold = df["model_score"].quantile(1 - MODEL_TOP_PCT)
model_flagged = (df["model_score"] >= model_threshold) & ~stale_visible & ~weak_ctr

conditions = [stale_visible, weak_ctr, model_flagged]
reason_codes = ["stale_but_visible", "weak_ctr_good_position", "model_flagged_decline_risk"]
action_labels = ["review_for_refresh", "review_for_ctr_fix", "review_general"]

df["reason_code"] = np.select(conditions, reason_codes, default="no_flag")
df["action_label"] = np.select(conditions, action_labels, default="no_action")

ranked = df.sort_values("model_score", ascending=False).reset_index(drop=True)

print("archetype counts:")
print(ranked["reason_code"].value_counts())

print("\ntop 15 of the ranked queue:")
print(ranked[["content_id", "reason_code", "action_label", "model_score",
              "days_since_last_update", "impressions_90d", "ctr", "position_tier"]].head(15))


archetype counts:
reason_code
no_flag                       19263
weak_ctr_good_position         5870
model_flagged_decline_risk     4850
stale_but_visible                17
Name: count, dtype: int64

top 15 of the ranked queue:
              content_id                 reason_code        action_label  model_score  days_since_last_update  \
0   content_1e446b05f1c5  model_flagged_decline_risk      review_general     0.838490                     104   
1   content_816431f05a3b  model_flagged_decline_risk      review_general     0.832871                     104   
2   content_3db2b454b5f7  model_flagged_decline_risk      review_general     0.832010                     104   
3   content_04d7435934f0  model_flagged_decline_risk      review_general     0.829450                     104   
4   content_8dba22b835f8  model_flagged_decline_risk      review_general     0.826522                     104   
5   content_f192f3938827  model_flagged_decline_risk      review_general     0.826082        

## 2. Intended use and limits

**Intended use:** a content/SEO team's triage aid for deciding which pages to look at first
each cycle. The ranking is **decision-support**, not an instruction — a human still opens each
flagged page and decides.

**What the evidence actually supports.** In this dataset, on a client-grouped holdout (Week 6),
the model's ranking was observed to be associated with `is_declining_label` at the precision@K
and ROC-AUC levels printed below — restated here, not re-measured on the in-sample scores used
for ranking in Section 1, since in-sample numbers would overstate confidence. The gap between
the random-split and grouped-split numbers from Week 6 is itself part of the evidence: it shows
how much of the model's apparent skill was genuine versus client memorization.

**Limits, stated plainly:**
- **Cross-sectional, one snapshot.** This is a single trailing-90-day window, not a time series.
  Nothing here supports "refreshing this page will cause traffic to recover" — only "pages that
  look like this were, in this snapshot, more often already declining." Correlational, not
  causal language throughout, per the claim ladder.
- **Tested on 32 clients, this slice only.** The Week-6 holdout is still drawn from the same 32
  pseudonymized clients as training — it tests generalization to unseen *pages*, not to a
  *new client* outside this population. A brand-new client's pages are outside what was
  validated.
- **Thin-volume tiers are noisy.** Per the data dictionary, `position_tier == top_3` runs on a
  low median search volume in this slice, where a single click swings CTR several points — the
  `weak_ctr_good_position` archetype should be read with that in mind, not taken at face value
  for low-volume pages.
- **Missing keyword data isn't random.** It follows `content_type` (e.g. `feedly article`), so
  the model's read on those rows leans more on activity/staleness signals and less on
  keyword-context signals than it does for other content types.


In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# restate the Week-6 honest (grouped-split) numbers as the evidence backing "intended use" above
# - NOT the in-sample model_score used for ranking in Section 1.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

eval_rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
eval_rf.fit(X_train, y_train)
eval_scores = eval_rf.predict_proba(X_test)[:, 1]

print("evidence for 'intended use' (Week-6 style honest, grouped-split holdout):")
print(f"  base rate (test): {y_test.mean():.3f}")
print(f"  ROC-AUC:          {roc_auc_score(y_test, eval_scores):.3f}")
for k in (20, 50, 200):
    print(f"  precision@{k}:    {precision_at_k(eval_scores, y_test.values, k):.3f}")


evidence for 'intended use' (Week-6 style honest, grouped-split holdout):
  base rate (test): 0.511
  ROC-AUC:          0.602
  precision@20:    0.550
  precision@50:    0.500
  precision@200:    0.455


## 3. Human review + the no-go list

**What a human must check before acting on any flagged row:**
- Does the page still match a live business priority? (A pseudonymized `content_id` carries no
  editorial or strategic context the model can see.)
- Is this page under legal, compliance, or brand constraints that make "refresh" inappropriate
  regardless of score (e.g. a policy page, a dated announcement)?
- Is a refresh already scheduled or recently completed outside this dataset's visibility?
- For `weak_ctr_good_position` rows: is the low CTR actually a title/meta problem, or a genuine
  mismatch between the page and current search intent that a title rewrite won't fix?
- For thin-volume rows (see Section 2): is there enough `n` behind the number to act on it at
  all, or is it noise?

**What should NOT be automated — no-go list:**
- No auto-publishing of rewritten titles, meta, or content from this ranking. A human drafts
  and approves; the model only orders the queue.
- No auto-deprioritizing or removing a page based solely on `model_score`. A low score here
  means "not flagged," not "safe to ignore" or "safe to delete."
- No client-facing promises tied to this ranking (e.g. "refreshing this will recover X% of
  traffic") — the evidence is correlational on one snapshot, not a causal, tested claim.
- No applying this ranking, unmodified, to a client outside the 32 in this dataset — see the
  generalization limit in Section 2.
- No treating `reason_code` as a complete explanation — it names the archetype that matched,
  not a guarantee that the archetype is the *true* cause of that page's situation.


In [5]:
# make the no-go/human-review guidance concrete with real numbers from the queue above.
review_flagged = ranked[ranked["action_label"] != "no_action"]
print(f"{len(review_flagged):,} of {len(ranked):,} rows flagged for human review ({len(review_flagged)/len(ranked):.1%})")

thin_volume_flag = review_flagged[
    (review_flagged["reason_code"] == "weak_ctr_good_position")
    & (review_flagged["position_tier"] == "top_3")
]
print(f"of those, {len(thin_volume_flag)} are top_3/weak-CTR rows - the thin-volume caveat from "
      f"Section 2 applies directly to these before anyone acts on them")

no_keyword_data = review_flagged[review_flagged["has_keyword_data"] == 0]
print(f"{len(no_keyword_data)} flagged rows have no keyword-context data at all "
      f"(content_type-linked missingness) - the model's read on these leans on activity/"
      f"staleness signals alone, worth a second look before acting")


10,737 of 30,000 rows flagged for human review (35.8%)
of those, 0 are top_3/weak-CTR rows - the thin-volume caveat from Section 2 applies directly to these before anyone acts on them
654 flagged rows have no keyword-context data at all (content_type-linked missingness) - the model's read on these leans on activity/staleness signals alone, worth a second look before acting


## 4. Monitoring / retrain triggers

**The decay/refresh insight, stated safely:** the two rule archetypes exist because staleness
and CTR-vs-position were **observed**, in the Week-4 and Week-6 signal checks, to be associated
with `is_declining_label` in this data — not because either causes decline. The playbook itself
will decay for the same underlying reason: it is fit to one 90-day snapshot's tiers and
thresholds, and search behavior, client mix, and the underlying trailing window all move.

**Light monitoring, practical rather than production-grade:**
- **Score/archetype drift** — track the archetype counts (Section 1's `value_counts()`) each
  time the queue is regenerated; a big shift in the mix (e.g. `model_flagged_decline_risk`
  suddenly dominating) is worth a look before trusting the new queue as-is.
- **Population drift** — track the overall base rate (`is_declining_label.mean()`); the current
  snapshot's 0.542 is the reference point this playbook was tuned against.
- **Human override rate** — if reviewers are routinely rejecting top-ranked items, that is a
  signal the ranking has drifted from what the team actually considers worth reviewing, even
  before any accuracy metric moves.

**Retrain triggers:**
- The underlying 90-day window rolls forward (new export of `content_refresh_anonymized.csv`)
  — retrain rather than keep scoring on a stale window.
- A held-out precision@K on a fresh grouped split drops meaningfully below the Week-6 reference
  numbers restated in Section 2.
- A new client is onboarded that the model has never seen — score with visible caution until a
  grouped holdout including that client exists.


In [6]:
# illustrative drift check - NOT production monitoring, just what a lightweight version looks like.
reference_base_rate = y.mean()
reference_archetype_counts = ranked["reason_code"].value_counts(normalize=True)

def drift_check(current_df, base_rate_tolerance=0.05, archetype_tolerance=0.10):
    current_base_rate = current_df["is_declining_label"].mean()
    current_archetype_counts = current_df["reason_code"].value_counts(normalize=True)

    base_rate_drift = abs(current_base_rate - reference_base_rate)
    archetype_drift = (current_archetype_counts - reference_archetype_counts).abs().max()

    print(f"base rate: reference={reference_base_rate:.3f}, current={current_base_rate:.3f}, "
          f"drift={base_rate_drift:.3f} ({'OK' if base_rate_drift <= base_rate_tolerance else 'FLAG - consider retraining'})")
    print(f"largest archetype share drift: {archetype_drift:.3f} "
          f"({'OK' if archetype_drift <= archetype_tolerance else 'FLAG - mix has shifted, inspect before trusting the queue'})")

# run it against this same run as a sanity check - on a future run, pass the NEW queue's dataframe here
drift_check(ranked)


base rate: reference=0.542, current=0.542, drift=0.000 (OK)
largest archetype share drift: 0.000 (OK)


## 5. Exports for the paper

The ranked queue goes to `work/outputs/` (not committed — CI's leak-guard blocks data files,
and the notebook regenerates it every run). The archetype-mix figure goes to `work/figures/`
(committed — this is what the paper's recommendations section will pull in). A metrics JSON
goes to `work/outputs/` too and **is** committed, per the card's instructions — it's the receipt
the paper's numbers trace back to, separate from the raw queue data.


In [7]:
import json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --- 1. the ranked queue (not committed - regenerated every run) ---
export_cols = [
    "content_id", "client_id", "model_score", "reason_code", "action_label",
    "days_since_last_update", "impressions_90d", "ctr", "avg_position", "position_tier",
]
ranked[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("wrote work/outputs/action_playbook_queue.csv")

# --- 2. archetype-mix figure (committed - reused in the paper) ---
fig, ax = plt.subplots(figsize=(6, 4))
ranked["reason_code"].value_counts().plot(kind="barh", ax=ax)
ax.set_xlabel("count")
ax.set_title("Action playbook: rows per archetype")
fig.tight_layout()
fig.savefig("work/figures/archetype_mix.png", dpi=150)
plt.close(fig)
print("wrote work/figures/archetype_mix.png")

# --- 3. metrics JSON (committed - the receipts the paper's numbers trace back to) ---
metrics = {
    "n_rows_total": int(len(ranked)),
    "n_rows_flagged": int((ranked["action_label"] != "no_action").sum()),
    "archetype_counts": ranked["reason_code"].value_counts().to_dict(),
    "reference_base_rate": float(reference_base_rate),
    "honest_holdout": {
        "split": "client-grouped, 80/20, random_state=42",
        "roc_auc": float(roc_auc_score(y_test, eval_scores)),
        "precision_at_20": float(precision_at_k(eval_scores, y_test.values, 20)),
        "precision_at_50": float(precision_at_k(eval_scores, y_test.values, 50)),
        "precision_at_200": float(precision_at_k(eval_scores, y_test.values, 200)),
    },
    "thresholds": {
        "stale_days_threshold": STALE_DAYS_THRESHOLD,
        "visible_impressions_threshold": VISIBLE_IMPRESSIONS_THRESHOLD,
        "model_top_pct": MODEL_TOP_PCT,
    },
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/w07_playbook_metrics.json")


wrote work/outputs/action_playbook_queue.csv
wrote work/figures/archetype_mix.png
wrote work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.